# Gradient boosting for classification
Eight students (CGPA, IQ, placed or not), step by step in log-odds, checked against scikit-learn's
`GradientBoostingClassifier`; then the same algorithm on a non-linear two-class dataset.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text

from gbc import boost, log_odds, sigmoid, two_gaussians

df = pd.read_csv("data/placement8.csv")
X, y = df[["cgpa", "iq"]].values, df["placed"].values
df

## Stage 1: log-odds, then probability

In [ ]:
f0 = np.log((y == 1).sum() / (y == 0).sum())       # log(5 / 3), natural log
print("log-odds:", round(f0, 4), " probability:", round(sigmoid(f0), 4))
df["F0"] = f0
df["p0"] = sigmoid(f0)
df["r1"] = y - df["p0"]                            # pseudo-residuals: actual minus probability
df

## Stage 2: a regression tree on the residuals, leaf values in log-odds

In [ ]:
tree1 = DecisionTreeRegressor(max_leaf_nodes=3, random_state=0).fit(X, df["r1"])
print(export_text(tree1, feature_names=["cgpa", "iq"]))
df["leaf1"] = tree1.apply(X)                       # leaf index of every row

def leaf_values(r, p, leaf):
    # sum of residuals / sum of p(1 - p), separately for each leaf
    return {j: r[leaf == j].sum() / (p[leaf == j] * (1 - p[leaf == j])).sum() for j in np.unique(leaf)}

g1 = leaf_values(df["r1"].values, df["p0"].values, df["leaf1"].values)
print({j: round(v, 4) for j, v in g1.items()})

In [ ]:
# combined model: add the leaf value to the log-odds, then turn back into a probability
df["F1"] = df["F0"] + df["leaf1"].map(g1)
df["p1"] = sigmoid(df["F1"])
df["r2"] = y - df["p1"]
df.round(4)

## Stage 3: a second tree on the new residuals

In [ ]:
tree2 = DecisionTreeRegressor(max_leaf_nodes=3, random_state=0).fit(X, df["r2"])
print(export_text(tree2, feature_names=["cgpa", "iq"]))
df["leaf2"] = tree2.apply(X)
g2 = leaf_values(df["r2"].values, df["p1"].values, df["leaf2"].values)
print({j: round(v, 4) for j, v in g2.items()})
df["F2"] = df["F1"] + df["leaf2"].map(g2)
df["p2"] = sigmoid(df["F2"])
df[["placed", "p0", "p1", "p2"]].round(4)

## Predicting a new student: CGPA 7.2, IQ 100

In [ ]:
q = np.array([[7.2, 100]])
z = f0 + g1[tree1.apply(q)[0]] + g2[tree2.apply(q)[0]]
print("log-odds:", round(z, 4), " probability:", round(sigmoid(z), 4))

## Check against scikit-learn

In [ ]:
# GradientBoostingClassifier starts from the log-odds and uses the same leaf formula
gbc = GradientBoostingClassifier(loss="log_loss", n_estimators=2, learning_rate=1.0,
                                 max_leaf_nodes=3, max_depth=None, random_state=0).fit(X, y)
for proba in gbc.staged_predict_proba(X):
    print(proba[:, 1].round(4))
print("new student:", gbc.predict_proba(q)[:, 1].round(4))

## With learning rate 0.1

In [ ]:
print((f0 + 0.1 * df["leaf1"].map(g1)).round(4).values)

## A non-linear dataset

In [ ]:
X2, y2 = two_gaussians()
X_train, X_test, y_train, y_test = train_test_split(X2, y2, test_size=0.25, random_state=42)
f0b, stages = boost(X_train, y_train, 100, learning_rate=0.5, max_leaf_nodes=4)
for m in (0, 1, 2, 10, 30, 100):
    acc = lambda Xs, ys: ((sigmoid(log_odds(f0b, stages, 0.5, Xs, m)) > 0.5) == ys).mean()
    print(f"{m:3d} trees: train {acc(X_train, y_train):.3f}, test {acc(X_test, y_test):.3f}")

In [ ]:
# our loop and scikit-learn agree at every stage
gbc = GradientBoostingClassifier(n_estimators=100, learning_rate=0.5, max_leaf_nodes=4,
                                 max_depth=None, random_state=42).fit(X_train, y_train)
gap = max(np.abs(sigmoid(log_odds(f0b, stages, 0.5, X_train, m)) - p[:, 1]).max()
          for m, p in enumerate(gbc.staged_predict_proba(X_train), start=1))
print(f"largest difference in probability: {gap:.1e}")